<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_J1_W7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge : Text Analysis of books using word cloud

Ce notebook présente une analyse complète des œuvres de Lewis Carroll en utilisant des techniques de Traitement du Langage Naturel (NLP).

In [ ]:
!pip install spacy wordcloud matplotlib pandas requests nltk sklearn
!python -m spacy download en_core_web_sm

In [ ]:
import requests
import re
import nltk
import spacy
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

nlp = spacy.load('en_core_web_sm')

### 1 & 2. Loading and Cleaning Texts
We use the Project Gutenberg URLs for Lewis Carroll's books.

In [ ]:
def load_texts(urls):
    corpus = []
    for url in urls:
        response = requests.get(url)
        text = response.text

        # Slicing to remove metadata
        start_marker = "*** START OF"
        end_marker = "*** END OF"

        start_idx = text.find(start_marker)
        if start_idx != -1:
            # Move index to after the line containing the marker
            start_idx = text.find("***", start_idx + 15) + 3

        end_idx = text.find(end_marker)
        if end_idx != -1:
            text = text[start_idx:end_idx]
        else:
            text = text[start_idx:]

        # Clean non-words using regex
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text)
        corpus.append(cleaned_text)
    return corpus

urls = [
    "https://www.gutenberg.org/files/11/11-0.txt", # Alice
    "https://www.gutenberg.org/files/12/12-0.txt", # Looking Glass
    "https://www.gutenberg.org/files/29042/29042-0.txt" # A Tangled Tale
]

book_names = ["Alice's Adventures in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]
raw_corpus = load_texts(urls)

for i, name in enumerate(book_names):
    print(f"--- {name} (First 200 chars) ---\n{raw_corpus[i][:200]}\n")

### 3, 4, 5 & 6. Preprocessing: Tokenization, Stopwords, Stemming, and Lemmatization

In [ ]:
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

processed_data = []

for text in raw_corpus:
    # 3. Tokenize
    tokens = nltk.word_tokenize(text.lower())

    # 4. Remove Stopwords
    filtered_tokens = [w for w in tokens if w not in stop_words]

    # 5. Stemming
    stemmed = [ps.stem(w) for w in filtered_tokens]

    # 6. Lemmatization (using spaCy on the original text to keep context)
    doc = nlp(text[:100000]) # Limit size for performance
    lemmatized = [token.lemma_ for token in doc if not token.is_stop and not token.is_space and not token.is_punct]

    processed_data.append({
        'tokens': tokens,
        'filtered': filtered_tokens,
        'stemmed': stemmed,
        'lemmatized': lemmatized
    })

print(f"First 150 tokens (Book 1): {processed_data[0]['tokens'][:150]}\n")
print(f"First 50 stemmed (Book 1): {processed_data[0]['stemmed'][:50]}\n")
print(f"First 50 lemmatized (Book 1): {processed_data[0]['lemmatized'][:50]}")

### 7. Analysis of Stemming vs Lemmatization
**Stemming** (Porter) uses heuristic rules to chop off ends of words (e.g., 'arguing' -> 'argu'). It is fast but can produce non-words.
**Lemmatization** (spaCy) uses a vocabulary and morphological analysis to return the base form ('arguing' -> 'argue'). It is more accurate but computationally heavier.

### 8 & 9. POS Tagging and NER

In [ ]:
for i in range(3):
    sample = " ".join(processed_data[i]['filtered'][:20])
    tokens = nltk.word_tokenize(sample)
    pos_tags = nltk.pos_tag(tokens)
    entities = nltk.ne_chunk(pos_tags)
    print(f"Book {i+1} POS Tags: {pos_tags}\n")

### Analyzing the text: WordCloud and BoW

In [ ]:
plt.figure(figsize=(15, 10))
for i in range(3):
    wc = WordCloud(background_color="white").generate(" ".join(processed_data[i]['lemmatized']))
    plt.subplot(1, 3, i+1)
    plt.imshow(wc)
    plt.title(book_names[i])
    plt.axis("off")
plt.show()

In [ ]:
# BoW for 5 most frequent words
vectorizer = CountVectorizer()
corpus_for_bow = [" ".join(d['lemmatized']) for d in processed_data]
X = vectorizer.fit_transform(corpus_for_bow)

feature_names = vectorizer.get_feature_names_out()
counts = X.toarray().sum(axis=0)

top_indices = counts.argsort()[-5:][::-1]
for idx in top_indices:
    print(f"Word: {feature_names[idx]}, Frequency: {counts[idx]}")

# Pie Chart
labels = [feature_names[idx] for idx in top_indices]
sizes = [counts[idx] for idx in top_indices]
plt.pie(sizes, labels=labels, autopct='%1.1f%%')
plt.title("Top 5 Words (BoW)")
plt.show()

### Solving the frequency problem using TF-IDF

In [ ]:
tfidf_vec = TfidfVectorizer(min_df=1, max_df=2)
X_tfidf = tfidf_vec.fit_transform(corpus_for_bow)
names_tfidf = tfidf_vec.get_feature_names_out()

for i in range(3):
    scores = X_tfidf[i].toarray().flatten()
    top_ids = scores.argsort()[-5:][::-1]

    plt.figure()
    plt.pie([scores[j] for j in top_ids], labels=[names_tfidf[j] for j in top_ids], autopct='%1.1f%%')
    plt.title(f"TF-IDF Top 5: {book_names[i]}")
    plt.show()